# 00_api_smoke_test

## 目的
- APIキーの読み込み確認
- XIT002 で市区町村リスト取得テスト
- XPT002 で数タイルの取得テスト
- 2026年データが取得可能か確認
- GeoJSON の中身確認

## 前提
- `.env` に `REINFOLIB_API_KEY` が設定されていること
- `pip install -r requirements.txt` が完了していること
- このノートブックは `notebooks/` から実行するか、カーネルの作業ディレクトリを親ディレクトリに設定してください

In [1]:
import sys, os
from pathlib import Path

# プロジェクトルートを config.py の存在で探索（CWD に依存しない）
def _find_project_root() -> str:
    # Jupyter が提供する _dh（notebook ディレクトリ）を優先
    try:
        _nb_dir = Path(globals()["_dh"][0])
    except (KeyError, IndexError):
        _nb_dir = Path.cwd()
    for candidate in [_nb_dir, _nb_dir.parent, _nb_dir.parent.parent]:
        if (candidate / "config.py").exists():
            return str(candidate.resolve())
    # フォールバック: 絶対パスで直接指定
    fallback = Path("/home/kazumasa/projects/land_price_api_app")
    if fallback.exists():
        return str(fallback)
    raise FileNotFoundError("プロジェクトルートが見つかりません")

_project_root = _find_project_root()
os.chdir(_project_root)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)
print("作業ディレクトリ:", os.getcwd())


作業ディレクトリ: /home/kazumasa/projects/land_price_api_app


## 1. APIキー確認

In [2]:
from api_client import smoke_test_api_key
ok = smoke_test_api_key()
print('APIキー:', '✓ OK' if ok else '✗ NG (.env を確認してください)')

2026-04-11 11:15:55,183 [INFO] api_client: APIキー: OK


APIキー: ✓ OK


## 2. XIT002 市区町村リスト取得テスト（東京都=13）

In [3]:
from api_client import fetch_city_list

cities = fetch_city_list('13')  # 東京都
print(f'東京都の市区町村: {len(cities)} 件')
for c in cities[:20]:
    print(' ', c)

東京都の市区町村: 62 件
  {'id': '13101', 'name': '千代田区'}
  {'id': '13102', 'name': '中央区'}
  {'id': '13103', 'name': '港区'}
  {'id': '13104', 'name': '新宿区'}
  {'id': '13105', 'name': '文京区'}
  {'id': '13106', 'name': '台東区'}
  {'id': '13107', 'name': '墨田区'}
  {'id': '13108', 'name': '江東区'}
  {'id': '13109', 'name': '品川区'}
  {'id': '13110', 'name': '目黒区'}
  {'id': '13111', 'name': '大田区'}
  {'id': '13112', 'name': '世田谷区'}
  {'id': '13113', 'name': '渋谷区'}
  {'id': '13114', 'name': '中野区'}
  {'id': '13115', 'name': '杉並区'}
  {'id': '13116', 'name': '豊島区'}
  {'id': '13117', 'name': '北区'}
  {'id': '13118', 'name': '荒川区'}
  {'id': '13119', 'name': '板橋区'}
  {'id': '13120', 'name': '練馬区'}


## 3. XPT002 タイル取得テスト（東京都心付近）

In [4]:
import pandas as pd
import plotly.express as px
from tiles import PREFECTURE_BBOX, prefecture_tile_range
from config import REQUEST_INTERVAL_SEC

Z = 13

PREF_NAMES = {
    "01": "北海道", "02": "青森", "03": "岩手", "04": "宮城", "05": "秋田",
    "06": "山形", "07": "福島", "08": "茨城", "09": "栃木", "10": "群馬",
    "11": "埼玉", "12": "千葉", "13": "東京", "14": "神奈川", "15": "新潟",
    "16": "富山", "17": "石川", "18": "福井", "19": "山梨", "20": "長野",
    "21": "岐阜", "22": "静岡", "23": "愛知", "24": "三重", "25": "滋賀",
    "26": "京都", "27": "大阪", "28": "兵庫", "29": "奈良", "30": "和歌山",
    "31": "鳥取", "32": "島根", "33": "岡山", "34": "広島", "35": "山口",
    "36": "徳島", "37": "香川", "38": "愛媛", "39": "高知", "40": "福岡",
    "41": "佐賀", "42": "長崎", "43": "熊本", "44": "大分", "45": "宮崎",
    "46": "鹿児島", "47": "沖縄",
}

rows = []
for code, name in PREF_NAMES.items():
    x_min, x_max, y_min, y_max = prefecture_tile_range(code, Z)
    n_tiles = (x_max - x_min + 1) * (y_max - y_min + 1)
    est_sec = n_tiles * REQUEST_INTERVAL_SEC
    rows.append({"コード": code, "都道府県": name, "タイル数": n_tiles, "推定秒": est_sec})

df_tiles = pd.DataFrame(rows).sort_values("タイル数", ascending=False)

total_tiles = df_tiles["タイル数"].sum()
total_min = df_tiles["推定秒"].sum() / 60

print(f"z={Z} 全国合計: {total_tiles:,} タイル / 推定 {total_min:.0f} 分（待機時間のみ、{REQUEST_INTERVAL_SEC}秒/タイル）")
print(df_tiles.to_string(index=False))

fig = px.bar(
    df_tiles,
    x="都道府県", y="タイル数",
    text="タイル数",
    title=f"都道府県別タイル数（z={Z}）／推定合計 {total_min:.0f} 分",
    color="タイル数",
    color_continuous_scale="Blues",
    height=500,
)
fig.update_traces(textposition="outside")
fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
fig.show()


z=13 全国合計: 65,380 タイル / 推定 327 分（待機時間のみ、0.3秒/タイル）
コード 都道府県  タイル数    推定秒
 01  北海道 15368 4610.4
 47   沖縄  9176 2752.8
 46  鹿児島  2499  749.7
 15   新潟  2352  705.6
 03   岩手  1782  534.6
 02   青森  1591  477.3
 07   福島  1584  475.2
 05   秋田  1395  418.5
 04   宮城  1365  409.5
 34   広島  1360  408.0
 42   長崎  1302  390.6
 21   岐阜  1225  367.5
 40   福岡  1188  356.4
 45   宮崎  1178  353.4
 39   高知  1056  316.8
 20   長野  1050  315.0
 08   茨城  1015  304.5
 06   山形  1014  304.2
 32   島根   930  279.0
 22   静岡   924  277.2
 18   福井   896  268.8
 24   三重   832  249.6
 35   山口   825  247.5
 44   大分   825  247.5
 28   兵庫   806  241.8
 43   熊本   784  235.2
 09   栃木   780  234.0
 38   愛媛   759  227.7
 10   群馬   754  226.2
 12   千葉   754  226.2
 36   徳島   713  213.9
 26   京都   672  201.6
 33   岡山   624  187.2
 41   佐賀   588  176.4
 23   愛知   552  165.6
 16   富山   546  163.8
 17   石川   513  153.9
 11   埼玉   504  151.2
 19   山梨   504  151.2
 25   滋賀   442  132.6
 31   鳥取   435  130.5
 14  神奈川   396  118.8
 30 

## 4. GeoJSON の中身確認

In [6]:
import json
from api_client import fetch_geojson_tile_xpt002

# 東京都心付近タイル (z=14, x=14552, y=6452)
TEST_Z, TEST_X, TEST_Y = 14, 14552, 6452
TARGET_YEAR = 2026

data = None
for try_year in (TARGET_YEAR, TARGET_YEAR - 1):
    try:
        data = fetch_geojson_tile_xpt002(TEST_Z, TEST_X, TEST_Y, year=try_year)
        features = data.get('features', [])
        print(f'year={try_year}: {len(features)} features 取得')
        if features:
            break
    except Exception as e:
        print(f'year={try_year}: 取得失敗 - {e}')

if not data or not data.get('features'):
    print('⚠ データが取得できませんでした。')
else:
    feat = data['features'][0]
    print('--- geometry ---')
    print(json.dumps(feat.get('geometry'), ensure_ascii=False, indent=2))
    print('--- properties (全キー) ---')
    for k, v in feat.get('properties', {}).items():
        print(f'  {k}: {v}')


year=2026: 33 features 取得
--- geometry ---
{
  "type": "Point",
  "coordinates": [
    139.75336253643036,
    35.67131691450372
  ]
}
--- properties (全キー) ---
  _id: -C0B9ZwBHMJXDabwWrxv
  _index: bi007_land_prices_2026-2026_202603161357
  location_number_ja: 内幸町２丁目２２番２
  area_division_name_ja: 
  sewer_supply_availability: False
  city_code: 13101
  residence_display_name_ja: 内幸町２－１－４
  side_road_name_ja: 
  u_road_distance_to_nearest_station_name_ja: 
  building_structure_name_ja: 
  regulations_use_category_name_ja: 
  side_road_azimuth_name_ja: 
  current_usage_status_of_surrounding_land_name_ja: 
  front_road_name_ja: 
  u_ground_hierarchy_ja: 
  use_category_name_ja: 商業地
  gas_supply_availability: False
  nearest_station_name_ja: 
  u_underground_hierarchy_ja: 
  year_on_year_change_rate: -
  u_regulations_building_coverage_ratio_ja: -(%)
  water_supply_availability: False
  u_cadastral_ja: 
  point_id: 3009006
  front_road_pavement_condition: 
  usage_category_name_ja: 
  prefe

## 5. 正規化テスト

In [7]:
from normalize import normalize_features_to_dataframe

if data and data.get('features'):
    df = normalize_features_to_dataframe(data['features'], year=try_year, price_classification=0)
    print(f'正規化後: {len(df)} 行 x {len(df.columns)} 列')
    display(df.head())
else:
    print('正規化するデータがありません')

正規化後: 33 行 x 30 列


,point_id,year,price_classification,survey_source,lon,lat,price_yen_per_sqm,area_sqm,building_coverage_ratio,floor_area_ratio,...,usage_status_name,building_structure_name,current_use_name,front_road_name,use_category_code,use_category_name,zoning_name,nearest_station_name,road_distance_to_station,front_road_condition
0,3009006,2026,0,地価公示,139.753363,35.671317,NaN,NaN,NaN,NaN,...,None,None,None,None,0,商業地,None,None,None,None
1,3009016,2026,0,地価公示,139.759145,35.672877,22000000.0,3850.0,80.0,900.0,...,劇場、事務所兼店舗等,SRC（鉄骨鉄筋 コンクリート造）,高層のホテル、劇場、事務所等が集まる商業地域,区道,0,商業地,商業地域,日比谷,100m,南西 18.0m 区道
2,3009020,2026,0,地価公示,139.760315,35.672860,5050000.0,87.0,80.0,900.0,...,店舗、事務所兼倉庫,S（鉄骨造）,中高層の飲食店舗ビル等が建ち並ぶ商業地域,区道,0,商業地,商業地域,日比谷,160m,南東 8.0m 区道
3,3009040,2026,0,地価公示,139.767337,35.672424,19700000.0,290.0,80.0,700.0,...,店舗,S（鉄骨造）,中高層の店舗、事務所ビルが多い商業地域,区道,0,商業地,商業地域,銀座,160m,北東 14.5m 区道
4,3009054,2026,0,地価公示,139.763995,35.672616,35600000.0,829.0,80.0,800.0,...,店舗兼事務所,SRC（鉄骨鉄筋 コンクリート造）,高層の店舗、事務所ビルが多い中心商業地域,都道,0,商業地,商業地域,銀座,0m,北西 30.0m 都道


## 6. 全カラム確認

In [8]:
if 'df' in dir() and not df.empty:
    print(df.dtypes)
    display(df.describe(include='all').T)

point_id                        object
year                             int64
price_classification             int64
survey_source                   object
lon                            float64
lat                            float64
price_yen_per_sqm              float64
area_sqm                       float64
building_coverage_ratio        float64
floor_area_ratio               float64
raw_properties                  object
standard_land_number            object
prefecture_code                 object
prefecture_name                 object
city_code                       object
city_name                       object
district_name                   object
location_text                   object
last_year_price_yen_per_sqm      int64
yoy_change_pct                 float64
usage_status_name               object
building_structure_name         object
current_use_name                object
front_road_name                 object
use_category_code                int64
use_category_name        

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
point_id,33,33,3009006,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,33.0,NaN,NaN,NaN,2026.0,0.0,2026.0,2026.0,2026.0,2026.0,2026.0
price_classification,33.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
survey_source,33,1,地価公示,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lon,33.0,NaN,NaN,NaN,139.758864,0.005279,139.749141,139.754494,139.757622,139.76348,139.767337
lat,33.0,NaN,NaN,NaN,35.667911,0.00437,35.658059,35.665952,35.668589,35.671317,35.674568
price_yen_per_sqm,31.0,NaN,NaN,NaN,17254838.709677,17157180.201686,2340000.0,6080000.0,11700000.0,17900000.0,67100000.0
area_sqm,31.0,NaN,NaN,NaN,870.516129,1330.108739,78.0,154.0,330.0,757.5,5518.0
building_coverage_ratio,31.0,NaN,NaN,NaN,80.0,0.0,80.0,80.0,80.0,80.0,80.0
floor_area_ratio,31.0,NaN,NaN,NaN,741.935484,92.282875,600.0,700.0,700.0,800.0,1000.0
